In [25]:
import os
import json
import time
import joblib
import requests
import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from requests.exceptions import Timeout, ConnectionError, RequestException
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
API_KEY = "1fd211478dd3b6374a813ec50561eb4e"

MODEL_DIR = "saved_best_models"

OPENWEATHER_AIR_URL = "http://api.openweathermap.org/data/2.5/air_pollution/history"
OPEN_METEO_ARCHIVE_URL = "https://api.open-meteo.com/v1/forecast"

TIMEZONE = "Asia/Makassar"
tz = ZoneInfo(TIMEZONE)

# Ambil data 24 jam ke belakang dari waktu sekarang
end_dt = datetime.now(tz)
start_dt = end_dt - timedelta(hours=24)

end_dt_naive = end_dt.replace(tzinfo=None)
start_dt_naive = start_dt.replace(tzinfo=None)

print("Start datetime:", start_dt_naive.strftime("%Y-%m-%d %H:%M:%S"))
print("End datetime  :", end_dt_naive.strftime("%Y-%m-%d %H:%M:%S"))

# Retry config
max_retries = 5
request_timeout = 30
retry_sleep = 10
sleep_seconds = 1.2

Start datetime: 2026-06-22 07:19:59
End datetime  : 2026-06-23 07:19:59


In [27]:
# =========================
# KOLOM POLUTAN DAN CUACA
# =========================

pollutant_cols = [
    "co", "no", "no2", "o3",
    "so2", "pm2_5", "pm10", "nh3"
]

weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "weather_code",
    "cloud_cover",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m"
]

lag_hours = [1, 3, 6, 12, 24]
rolling_windows = [3, 6, 24]

In [28]:
# =========================
# FUNGSI FETCH DENGAN RETRY
# =========================

def fetch_with_retry(
    url,
    params,
    max_retries=5,
    timeout=30,
    retry_sleep=10,
    last_success_dt=None
):
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, params=params, timeout=timeout)

            print("      Status code:", response.status_code)

            if response.status_code == 200:
                return response.json()

            print(f"      Status bukan 200. Percobaan {attempt}/{max_retries}")
            print("      Response:", response.text[:500])

        except Timeout:
            print(f"      Timeout. Percobaan {attempt}/{max_retries}")

            if last_success_dt is not None:
                print(f"      Data terakhir berhasil diambil sampai: {last_success_dt}")
            else:
                print("      Belum ada data yang berhasil diambil.")

        except ConnectionError as e:
            print(f"      Koneksi gagal. Percobaan {attempt}/{max_retries}")

            if last_success_dt is not None:
                print(f"      Data terakhir berhasil diambil sampai: {last_success_dt}")
            else:
                print("      Belum ada data yang berhasil diambil.")

            print("      Detail error:", e)

        except RequestException as e:
            print(f"      Request error. Percobaan {attempt}/{max_retries}")

            if last_success_dt is not None:
                print(f"      Data terakhir berhasil diambil sampai: {last_success_dt}")
            else:
                print("      Belum ada data yang berhasil diambil.")

            print("      Detail error:", e)

        if attempt < max_retries:
            print(f"      Menunggu {retry_sleep} detik sebelum retry...")
            time.sleep(retry_sleep)

    return None

In [29]:
# =========================
# BACA FILE JSON KOORDINAT
# =========================

with open("koordinat.json", "r", encoding="utf-8") as f:
    regions = json.load(f)

print("File koordinat.json berhasil dibaca.")

total_points = sum(
    len(points)
    for districts in regions.values()
    for points in districts.values()
)

print("Total kabupaten/kota:", len(regions))
print("Total titik:", total_points)

File koordinat.json berhasil dibaca.
Total kabupaten/kota: 8
Total titik: 588


In [30]:
# =========================
# FLATTEN KOORDINAT JADI LIST TITIK
# =========================

all_points = []

for regency_name, districts in regions.items():
    for district_name, points in districts.items():
        for point_index, point in enumerate(points, start=1):
            all_points.append({
                "regency": regency_name,
                "district": district_name,
                "point_index": point_index,
                "total_points_in_district": len(points),
                "lat": point["lat"],
                "lon": point["lon"],
                "description": point.get("description", ""),
                "point_id": f"{regency_name}_{district_name}_{point_index}"
            })

print("Total titik siap diproses:", len(all_points))

Total titik siap diproses: 588


In [31]:
# =========================
# SCRAPING OPENWEATHER AIR POLLUTION
# 24 JAM TERAKHIR
# =========================

BATCH_SIZE = 10

start_unix = int(start_dt.timestamp())
end_unix = int(end_dt.timestamp())

def fetch_air_point(point):
    lat = point["lat"]
    lon = point["lon"]

    params = {
        "lat": lat,
        "lon": lon,
        "start": start_unix,
        "end": end_unix,
        "appid": API_KEY
    }

    print(
        f"Ambil AQ | {point['regency']} - {point['district']} "
        f"| Titik {point['point_index']} | {lat}, {lon}"
    )

    data = fetch_with_retry(
        url=OPENWEATHER_AIR_URL,
        params=params,
        max_retries=max_retries,
        timeout=request_timeout,
        retry_sleep=retry_sleep,
        last_success_dt=None
    )

    rows = []

    if data is None:
        print(f"  Gagal AQ: {point['point_id']}")
        return rows

    if "list" not in data:
        print(f"  Response AQ tidak memiliki key 'list': {point['point_id']}")
        print("  Response:", data)
        return rows

    if len(data["list"]) == 0:
        print(f"  Data AQ kosong: {point['point_id']}")
        return rows

    for item in data["list"]:
        components = item["components"]

        timestamp = datetime.fromtimestamp(
            item["dt"],
            tz=tz
        ).replace(tzinfo=None)

        rows.append({
            "timestamp": timestamp,
            "regency": point["regency"],
            "district": point["district"],
            "point_id": point["point_id"],
            "description": point["description"],
            "lat": lat,
            "lon": lon,
            "aqi": item["main"]["aqi"],
            "co": components.get("co"),
            "no": components.get("no"),
            "no2": components.get("no2"),
            "o3": components.get("o3"),
            "so2": components.get("so2"),
            "pm2_5": components.get("pm2_5"),
            "pm10": components.get("pm10"),
            "nh3": components.get("nh3")
        })

    if rows:
        print(f"  Berhasil AQ: {point['point_id']} | data: {len(rows)}")

    return rows


air_rows = []

for batch_start in range(0, len(all_points), BATCH_SIZE):
    batch_points = all_points[batch_start:batch_start + BATCH_SIZE]

    print(
        f"\nMemproses batch AQ "
        f"{batch_start + 1} - {batch_start + len(batch_points)} "
        f"dari {len(all_points)} titik"
    )

    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        futures = [
            executor.submit(fetch_air_point, point)
            for point in batch_points
        ]

        for future in as_completed(futures):
            try:
                rows = future.result()
                air_rows.extend(rows)
            except Exception as e:
                print("Error saat mengambil data AQ:", e)

    time.sleep(sleep_seconds)

air_df = pd.DataFrame(air_rows)

print("\nJumlah data kualitas udara:", len(air_df))
display(air_df.head())


Memproses batch AQ 1 - 10 dari 588 titik
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 1 | -8.6516353, 115.1947762
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 2 | -8.681709, 115.19703
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 3 | -8.654969, 115.2103865
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 4 | -8.676031, 115.215224
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 5 | -8.684573, 115.226236
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 6 | -8.679993, 115.180557
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 7 | -8.700204, 115.185217
Ambil AQ | Kota Denpasar - Denpasar Barat | Titik 8 | -8.648762, 115.186305
Ambil AQ | Kota Denpasar - Denpasar Timur | Titik 1 | -8.6483648, 115.2376801
Ambil AQ | Kota Denpasar - Denpasar Timur | Titik 2 | -8.649475, 115.255338
      Status code: 200
  Berhasil AQ: Kota Denpasar_Denpasar Barat_5 | data: 24
      Status code: 200
  Berhasil AQ: Kota Denpasar_Denpasar Barat_3 | data: 24
      Status code: 200
  Berhasil 

,timestamp,regency,district,point_id,description,lat,lon,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2026-06-22 08:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_5,"Jalan Tukad Pakerisan, INSTIKI",-8.684573,115.226236,1,69.45,0.01,0.09,35.76,0.15,4.07,11.70,0.10
1,2026-06-22 09:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_5,"Jalan Tukad Pakerisan, INSTIKI",-8.684573,115.226236,1,70.50,0.02,0.08,35.52,0.15,3.96,11.16,0.12
2,2026-06-22 10:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_5,"Jalan Tukad Pakerisan, INSTIKI",-8.684573,115.226236,1,71.52,0.02,0.07,35.52,0.15,3.95,11.19,0.16
3,2026-06-22 11:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_5,"Jalan Tukad Pakerisan, INSTIKI",-8.684573,115.226236,1,72.23,0.02,0.07,35.44,0.15,4.04,11.96,0.19
4,2026-06-22 12:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_5,"Jalan Tukad Pakerisan, INSTIKI",-8.684573,115.226236,1,72.69,0.02,0.06,35.33,0.15,4.19,13.21,0.24


In [32]:
# =========================
# SCRAPING OPEN-METEO FORECAST API
# 24 JAM TERAKHIR SAMPAI CURRENT DATETIME
# KOLOM CUACA SESUAI TRAINING
# =========================

BATCH_SIZE = 10

weather_df_list = []

print("Weather start_dt:", start_dt)
print("Weather end_dt  :", end_dt)
print("Weather cols    :", weather_cols)


def fetch_weather_point(point):
    lat = point["lat"]
    lon = point["lon"]

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ",".join(weather_cols),
        "timezone": TIMEZONE,
        "past_days": 1,
        "forecast_days": 1
    }

    print(
        f"Ambil Weather | {point['regency']} - {point['district']} "
        f"| Titik {point['point_index']} | {lat}, {lon}"
    )

    data = fetch_with_retry(
        url=OPEN_METEO_ARCHIVE_URL,
        params=params,
        max_retries=max_retries,
        timeout=request_timeout,
        retry_sleep=retry_sleep,
        last_success_dt=None
    )

    if data is None:
        print(f"  Gagal Weather: {point['point_id']}")
        return pd.DataFrame()

    if "hourly" not in data:
        print(f"  Response Weather tidak memiliki key 'hourly': {point['point_id']}")
        print("  Response:", data)
        return pd.DataFrame()

    weather_point_df = pd.DataFrame(data["hourly"])

    if weather_point_df.empty:
        print(f"  Data Weather kosong: {point['point_id']}")
        return pd.DataFrame()

    weather_point_df["timestamp"] = pd.to_datetime(weather_point_df["time"])

    if weather_point_df["timestamp"].dt.tz is None:
        weather_point_df["timestamp"] = weather_point_df["timestamp"].dt.tz_localize(None)

    # Filter hanya 24 jam terakhir sampai current datetime
    weather_point_df = weather_point_df[
        (weather_point_df["timestamp"] >= start_dt_naive) &
        (weather_point_df["timestamp"] <= end_dt_naive)
    ].copy()

    if weather_point_df.empty:
        print(f"  Data Weather kosong setelah filter 24 jam: {point['point_id']}")
        return pd.DataFrame()

    weather_point_df["regency"] = point["regency"]
    weather_point_df["district"] = point["district"]
    weather_point_df["point_id"] = point["point_id"]
    weather_point_df["description"] = point["description"]
    weather_point_df["lat"] = lat
    weather_point_df["lon"] = lon

    print(f"  Berhasil Weather: {point['point_id']} | data: {len(weather_point_df)}")

    return weather_point_df


for batch_start in range(0, len(all_points), BATCH_SIZE):
    batch_points = all_points[batch_start:batch_start + BATCH_SIZE]

    print(
        f"\nMemproses batch Weather "
        f"{batch_start + 1} - {batch_start + len(batch_points)} "
        f"dari {len(all_points)} titik"
    )

    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        futures = [
            executor.submit(fetch_weather_point, point)
            for point in batch_points
        ]

        for future in as_completed(futures):
            try:
                weather_point_df = future.result()

                if not weather_point_df.empty:
                    weather_df_list.append(weather_point_df)

            except Exception as e:
                print("Error saat mengambil data Weather:", e)

    time.sleep(sleep_seconds)


if len(weather_df_list) > 0:
    weather_df = pd.concat(weather_df_list, ignore_index=True)
else:
    weather_df = pd.DataFrame()

print("\nJumlah data cuaca:", len(weather_df))
display(weather_df.head())

Weather start_dt: 2026-06-22 07:19:59.702720+08:00
Weather end_dt  : 2026-06-23 07:19:59.702720+08:00
Weather cols    : ['temperature_2m', 'relative_humidity_2m', 'apparent_temperature', 'precipitation', 'rain', 'weather_code', 'cloud_cover', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m']

Memproses batch Weather 1 - 10 dari 588 titik
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 1 | -8.6516353, 115.1947762
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 2 | -8.681709, 115.19703
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 3 | -8.654969, 115.2103865
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 4 | -8.676031, 115.215224
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 5 | -8.684573, 115.226236
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 6 | -8.679993, 115.180557
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 7 | -8.700204, 115.185217
Ambil Weather | Kota Denpasar - Denpasar Barat | Titik 8 | -8.6

,time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,cloud_cover,surface_pressure,wind_speed_10m,wind_direction_10m,wind_gusts_10m,timestamp,regency,district,point_id,description,lat,lon
0,2026-06-22T08:00,25.1,88,29.6,0.0,0.0,0,8,1010.3,6.3,35,15.5,2026-06-22 08:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_3,Pasar Badung,-8.654969,115.210386
1,2026-06-22T09:00,27.3,79,32.0,0.0,0.0,0,4,1010.7,6.8,87,18.0,2026-06-22 09:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_3,Pasar Badung,-8.654969,115.210386
2,2026-06-22T10:00,28.6,72,32.9,0.0,0.0,0,3,1010.2,8.3,108,22.7,2026-06-22 10:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_3,Pasar Badung,-8.654969,115.210386
3,2026-06-22T11:00,29.5,67,34.4,0.0,0.0,0,3,1009.7,8.9,122,24.8,2026-06-22 11:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_3,Pasar Badung,-8.654969,115.210386
4,2026-06-22T12:00,29.9,67,35.2,0.0,0.0,0,2,1008.8,10.8,134,28.8,2026-06-22 12:00:00,Kota Denpasar,Denpasar Barat,Kota Denpasar_Denpasar Barat_3,Pasar Badung,-8.654969,115.210386


In [33]:
# =========================
# RATA-RATA KUALITAS UDARA
# TITIK → KECAMATAN → KABUPATEN/KOTA
# PER JAM
# =========================

if air_df.empty:
    raise ValueError("air_df kosong. Tidak bisa lanjut ke proses rata-rata dan prediksi.")

air_df["timestamp"] = pd.to_datetime(air_df["timestamp"])
air_df["timestamp_hour"] = air_df["timestamp"].dt.floor("h")

# =========================
# 1. RATA-RATA TITIK → KECAMATAN PER JAM
# =========================

air_district_hourly_avg = (
    air_df
    .groupby(
        ["timestamp_hour", "regency", "district"],
        as_index=False
    )[pollutant_cols]
    .mean()
)

air_district_hourly_avg = air_district_hourly_avg.sort_values(
    by=["timestamp_hour", "regency", "district"]
).reset_index(drop=True)

print("Rata-rata kualitas udara titik → kecamatan per jam selesai.")
print("Total baris kecamatan:", len(air_district_hourly_avg))
display(air_district_hourly_avg.head())


# =========================
# 2. RATA-RATA KECAMATAN → KABUPATEN/KOTA PER JAM
# Metode: equal district average
# Setiap kecamatan punya bobot sama
# =========================

air_regency_hourly_avg = (
    air_district_hourly_avg
    .groupby(
        ["timestamp_hour", "regency"],
        as_index=False
    )[pollutant_cols]
    .mean()
)

air_regency_hourly_avg = air_regency_hourly_avg.sort_values(
    by=["timestamp_hour", "regency"]
).reset_index(drop=True)

print("Rata-rata kualitas udara kecamatan → kabupaten/kota per jam selesai.")
print("Total baris kabupaten/kota:", len(air_regency_hourly_avg))
display(air_regency_hourly_avg.head())

Rata-rata kualitas udara titik → kecamatan per jam selesai.
Total baris kecamatan: 1368


,timestamp_hour,regency,district,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2026-06-22 08:00:00,Kabupaten Badung,Abiansemal,70.450000,0.010000,0.100000,36.052308,0.16,3.974615,11.993846,0.126923
1,2026-06-22 08:00:00,Kabupaten Badung,Kuta,68.990000,0.010000,0.080000,35.800000,0.15,4.210000,12.320000,0.090000
2,2026-06-22 08:00:00,Kabupaten Badung,Kuta Selatan,68.544615,0.004615,0.077692,35.642308,0.15,4.263846,12.166923,0.076923
3,2026-06-22 08:00:00,Kabupaten Badung,Kuta Utara,68.990000,0.010000,0.080000,35.800000,0.15,4.210000,12.320000,0.090000
4,2026-06-22 08:00:00,Kabupaten Badung,Mengwi,70.090000,0.010000,0.100000,36.080000,0.16,4.030000,12.250000,0.120000


Rata-rata kualitas udara kecamatan → kabupaten/kota per jam selesai.
Total baris kabupaten/kota: 192


,timestamp_hour,regency,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2026-06-22 08:00:00,Kabupaten Badung,69.545769,0.009103,0.089615,35.907564,0.155000,4.116667,12.202564,0.104359
1,2026-06-22 08:00:00,Kabupaten Bangli,73.026731,0.013077,0.146346,36.477692,0.163462,3.851346,11.339808,0.210192
2,2026-06-22 08:00:00,Kabupaten Buleleng,84.846410,0.017607,0.323675,39.287521,0.155556,3.328120,8.291111,0.562735
3,2026-06-22 08:00:00,Kabupaten Gianyar,71.769157,0.010706,0.125147,36.370157,0.159402,3.939902,11.735216,0.171343
4,2026-06-22 08:00:00,Kabupaten Jembrana,76.077231,0.010769,0.161692,37.620923,0.152000,3.849846,10.997077,0.263692


In [34]:
# =========================
# RATA-RATA CUACA
# TITIK → KECAMATAN → KABUPATEN/KOTA
# PER JAM
# =========================

if weather_df.empty:
    print("weather_df kosong. Data cuaca tidak akan digunakan.")

    weather_district_hourly_avg = pd.DataFrame(
        columns=["timestamp_hour", "regency", "district"] + weather_cols
    )

    weather_regency_hourly_avg = pd.DataFrame(
        columns=["timestamp_hour", "regency"] + weather_cols
    )

else:
    weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])
    weather_df["timestamp_hour"] = weather_df["timestamp"].dt.floor("h")

    # =========================
    # 1. RATA-RATA TITIK → KECAMATAN PER JAM
    # =========================

    weather_district_hourly_avg = (
        weather_df
        .groupby(
            ["timestamp_hour", "regency", "district"],
            as_index=False
        )[weather_cols]
        .mean()
    )

    weather_district_hourly_avg = weather_district_hourly_avg.sort_values(
        by=["timestamp_hour", "regency", "district"]
    ).reset_index(drop=True)

    print("Rata-rata cuaca titik → kecamatan per jam selesai.")
    print("Total baris kecamatan:", len(weather_district_hourly_avg))
    display(weather_district_hourly_avg.head())


    # =========================
    # 2. RATA-RATA KECAMATAN → KABUPATEN/KOTA PER JAM
    # Metode: equal district average
    # Setiap kecamatan punya bobot sama
    # =========================

    weather_regency_hourly_avg = (
        weather_district_hourly_avg
        .groupby(
            ["timestamp_hour", "regency"],
            as_index=False
        )[weather_cols]
        .mean()
    )

    weather_regency_hourly_avg = weather_regency_hourly_avg.sort_values(
        by=["timestamp_hour", "regency"]
    ).reset_index(drop=True)

    print("Rata-rata cuaca kecamatan → kabupaten/kota per jam selesai.")
    print("Total baris kabupaten/kota:", len(weather_regency_hourly_avg))
    display(weather_regency_hourly_avg.head())

Rata-rata cuaca titik → kecamatan per jam selesai.
Total baris kecamatan: 1368


,timestamp_hour,regency,district,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,cloud_cover,surface_pressure,wind_speed_10m,wind_direction_10m,wind_gusts_10m
0,2026-06-22 08:00:00,Kabupaten Badung,Abiansemal,24.476923,86.461538,28.553846,0.0,0.0,0.0,17.076923,999.653846,6.169231,53.846154,14.023077
1,2026-06-22 08:00:00,Kabupaten Badung,Kuta,25.346154,88.000000,30.000000,0.0,0.0,0.0,8.000000,1013.107692,6.300000,35.000000,15.500000
2,2026-06-22 08:00:00,Kabupaten Badung,Kuta Selatan,27.084615,78.692308,30.792308,0.0,0.0,0.0,4.615385,1007.369231,12.392308,70.538462,15.500000
3,2026-06-22 08:00:00,Kabupaten Badung,Kuta Utara,25.346154,86.153846,29.461538,0.0,0.0,0.0,7.461538,1011.753846,8.400000,58.307692,17.576923
4,2026-06-22 08:00:00,Kabupaten Badung,Mengwi,24.730769,84.923077,28.476923,0.0,0.0,0.0,9.000000,999.492308,8.230769,70.923077,16.053846


Rata-rata cuaca kecamatan → kabupaten/kota per jam selesai.
Total baris kabupaten/kota: 192


,timestamp_hour,regency,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,cloud_cover,surface_pressure,wind_speed_10m,wind_direction_10m,wind_gusts_10m
0,2026-06-22 08:00:00,Kabupaten Badung,25.138462,85.064103,29.188462,0.0,0.0,0.102564,10.653846,1001.288462,7.547436,68.487179,14.862821
1,2026-06-22 08:00:00,Kabupaten Bangli,21.751923,89.326923,24.755769,0.0,0.0,0.692308,29.230769,935.742308,5.646154,38.461538,14.438462
2,2026-06-22 08:00:00,Kabupaten Buleleng,27.061538,73.239316,31.188889,0.0,0.0,0.162393,11.179487,1002.635897,4.431624,187.025641,12.666667
3,2026-06-22 08:00:00,Kabupaten Gianyar,23.920035,87.210504,27.805602,0.0,0.0,0.491176,19.084384,979.983746,5.560385,84.142367,13.146653
4,2026-06-22 08:00:00,Kabupaten Jembrana,25.924615,83.061538,30.547692,0.0,0.0,0.030769,5.769231,1006.595385,4.783077,91.000000,12.363077


In [35]:
# =========================
# CEK MISSING TIMESTAMP
# =========================

def check_missing_timestamp_per_regency(
    df,
    timestamp_col="timestamp_hour",
    freq="1h"
):
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])

    missing_report = []

    for regency, group in df.groupby("regency"):
        group = group.sort_values(timestamp_col)

        existing_times = pd.DatetimeIndex(
            group[timestamp_col].drop_duplicates()
        )

        if existing_times.empty:
            continue

        full_range = pd.date_range(
            start=existing_times.min(),
            end=existing_times.max(),
            freq=freq
        )

        missing_times = full_range.difference(existing_times)

        if len(missing_times) > 0:
            missing_report.append({
                "regency": regency,
                "start_time": existing_times.min(),
                "end_time": existing_times.max(),
                "expected_rows": len(full_range),
                "actual_rows": len(existing_times),
                "missing_count": len(missing_times),
                "missing_timestamps": list(missing_times)
            })

    return pd.DataFrame(missing_report)


def complete_timestamp_per_regency(
    df,
    value_cols,
    timestamp_col="timestamp_hour",
    freq="1h"
):
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])

    completed_list = []

    for regency, group in df.groupby("regency"):
        group = group.sort_values(timestamp_col)

        # Hapus duplikasi timestamp-regency agar reindex tidak error
        group = group.drop_duplicates(
            subset=[timestamp_col, "regency"]
        )

        full_range = pd.date_range(
            start=group[timestamp_col].min(),
            end=group[timestamp_col].max(),
            freq=freq
        )

        group = group.set_index(timestamp_col)
        group = group.reindex(full_range)

        group.index.name = timestamp_col

        # Isi kembali nama regency untuk baris timestamp yang baru dibuat
        group["regency"] = regency

        # Kolom nilai yang benar-benar ada di dataframe
        existing_value_cols = [
            col for col in value_cols
            if col in group.columns
        ]

        # Isi nilai yang hilang dengan interpolasi waktu
        group[existing_value_cols] = (
            group[existing_value_cols]
            .interpolate(method="time", limit_direction="both")
            .ffill()
            .bfill()
        )

        completed_list.append(group.reset_index())

    if len(completed_list) == 0:
        return df

    return pd.concat(completed_list, ignore_index=True)

In [36]:
# =========================
# LENGKAPI AIR POLLUTION
# =========================

if air_regency_hourly_avg.empty:
    raise ValueError("air_regency_hourly_avg kosong. Tidak bisa cek missing timestamp.")

air_missing_report = check_missing_timestamp_per_regency(
    df=air_regency_hourly_avg,
    timestamp_col="timestamp_hour",
    freq="1h"
)

if air_missing_report.empty:
    print("Tidak ada missing timestamp pada air pollution level kabupaten/kota.")
    air_regency_hourly_completed = air_regency_hourly_avg.copy()
else:
    print("Ditemukan missing timestamp pada air pollution level kabupaten/kota.")
    print("Jumlah kabupaten/kota bermasalah:", len(air_missing_report))

    display(
        air_missing_report[
            [
                "regency",
                "start_time",
                "end_time",
                "expected_rows",
                "actual_rows",
                "missing_count"
            ]
        ]
    )

    air_regency_hourly_completed = complete_timestamp_per_regency(
        df=air_regency_hourly_avg,
        value_cols=pollutant_cols,
        timestamp_col="timestamp_hour",
        freq="1h"
    )

print("Ukuran air sebelum dilengkapi:", air_regency_hourly_avg.shape)
print("Ukuran air setelah dilengkapi:", air_regency_hourly_completed.shape)

display(air_regency_hourly_completed.head())

Tidak ada missing timestamp pada air pollution level kabupaten/kota.
Ukuran air sebelum dilengkapi: (192, 10)
Ukuran air setelah dilengkapi: (192, 10)


,timestamp_hour,regency,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2026-06-22 08:00:00,Kabupaten Badung,69.545769,0.009103,0.089615,35.907564,0.155000,4.116667,12.202564,0.104359
1,2026-06-22 08:00:00,Kabupaten Bangli,73.026731,0.013077,0.146346,36.477692,0.163462,3.851346,11.339808,0.210192
2,2026-06-22 08:00:00,Kabupaten Buleleng,84.846410,0.017607,0.323675,39.287521,0.155556,3.328120,8.291111,0.562735
3,2026-06-22 08:00:00,Kabupaten Gianyar,71.769157,0.010706,0.125147,36.370157,0.159402,3.939902,11.735216,0.171343
4,2026-06-22 08:00:00,Kabupaten Jembrana,76.077231,0.010769,0.161692,37.620923,0.152000,3.849846,10.997077,0.263692


In [37]:
# =========================
# LENGKAPI WEATHER
# =========================

if weather_regency_hourly_avg.empty:
    print("weather_regency_hourly_avg kosong. Lewati cek missing timestamp weather.")

    weather_regency_hourly_completed = pd.DataFrame(
        columns=["timestamp_hour", "regency"] + weather_cols
    )

else:
    weather_missing_report = check_missing_timestamp_per_regency(
        df=weather_regency_hourly_avg,
        timestamp_col="timestamp_hour",
        freq="1h"
    )

    if weather_missing_report.empty:
        print("Tidak ada missing timestamp pada weather level kabupaten/kota.")
        weather_regency_hourly_completed = weather_regency_hourly_avg.copy()
    else:
        print("Ditemukan missing timestamp pada weather level kabupaten/kota.")
        print("Jumlah kabupaten/kota bermasalah:", len(weather_missing_report))

        display(
            weather_missing_report[
                [
                    "regency",
                    "start_time",
                    "end_time",
                    "expected_rows",
                    "actual_rows",
                    "missing_count"
                ]
            ]
        )

        weather_regency_hourly_completed = complete_timestamp_per_regency(
            df=weather_regency_hourly_avg,
            value_cols=weather_cols,
            timestamp_col="timestamp_hour",
            freq="1h"
        )

    print("Ukuran weather sebelum dilengkapi:", weather_regency_hourly_avg.shape)
    print("Ukuran weather setelah dilengkapi:", weather_regency_hourly_completed.shape)

    display(weather_regency_hourly_completed.head())

Tidak ada missing timestamp pada weather level kabupaten/kota.
Ukuran weather sebelum dilengkapi: (192, 13)
Ukuran weather setelah dilengkapi: (192, 13)


,timestamp_hour,regency,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,cloud_cover,surface_pressure,wind_speed_10m,wind_direction_10m,wind_gusts_10m
0,2026-06-22 08:00:00,Kabupaten Badung,25.138462,85.064103,29.188462,0.0,0.0,0.102564,10.653846,1001.288462,7.547436,68.487179,14.862821
1,2026-06-22 08:00:00,Kabupaten Bangli,21.751923,89.326923,24.755769,0.0,0.0,0.692308,29.230769,935.742308,5.646154,38.461538,14.438462
2,2026-06-22 08:00:00,Kabupaten Buleleng,27.061538,73.239316,31.188889,0.0,0.0,0.162393,11.179487,1002.635897,4.431624,187.025641,12.666667
3,2026-06-22 08:00:00,Kabupaten Gianyar,23.920035,87.210504,27.805602,0.0,0.0,0.491176,19.084384,979.983746,5.560385,84.142367,13.146653
4,2026-06-22 08:00:00,Kabupaten Jembrana,25.924615,83.061538,30.547692,0.0,0.0,0.030769,5.769231,1006.595385,4.783077,91.000000,12.363077


In [38]:
# =========================
# GABUNGKAN KUALITAS UDARA + CUACA
# LEVEL KABUPATEN/KOTA PER JAM
# =========================

regency_hourly_input = air_regency_hourly_completed.merge(
    weather_regency_hourly_completed,
    on=["timestamp_hour", "regency"],
    how="left"
)

regency_hourly_input = regency_hourly_input.sort_values(
    by=["timestamp_hour", "regency"]
).reset_index(drop=True)

print("Merge data air pollution dan weather selesai.")
print("Total baris:", len(regency_hourly_input))

display(regency_hourly_input.head())

Merge data air pollution dan weather selesai.
Total baris: 192


,timestamp_hour,regency,co,no,no2,o3,so2,pm2_5,pm10,nh3,...,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,cloud_cover,surface_pressure,wind_speed_10m,wind_direction_10m,wind_gusts_10m
0,2026-06-22 08:00:00,Kabupaten Badung,69.545769,0.009103,0.089615,35.907564,0.155000,4.116667,12.202564,0.104359,...,85.064103,29.188462,0.0,0.0,0.102564,10.653846,1001.288462,7.547436,68.487179,14.862821
1,2026-06-22 08:00:00,Kabupaten Bangli,73.026731,0.013077,0.146346,36.477692,0.163462,3.851346,11.339808,0.210192,...,89.326923,24.755769,0.0,0.0,0.692308,29.230769,935.742308,5.646154,38.461538,14.438462
2,2026-06-22 08:00:00,Kabupaten Buleleng,84.846410,0.017607,0.323675,39.287521,0.155556,3.328120,8.291111,0.562735,...,73.239316,31.188889,0.0,0.0,0.162393,11.179487,1002.635897,4.431624,187.025641,12.666667
3,2026-06-22 08:00:00,Kabupaten Gianyar,71.769157,0.010706,0.125147,36.370157,0.159402,3.939902,11.735216,0.171343,...,87.210504,27.805602,0.0,0.0,0.491176,19.084384,979.983746,5.560385,84.142367,13.146653
4,2026-06-22 08:00:00,Kabupaten Jembrana,76.077231,0.010769,0.161692,37.620923,0.152000,3.849846,10.997077,0.263692,...,83.061538,30.547692,0.0,0.0,0.030769,5.769231,1006.595385,4.783077,91.000000,12.363077


In [39]:
# =========================
# FEATURE ENGINEERING
# TIME, MUSIM, LAG, ROLLING
# =========================

final_input_data = regency_hourly_input.copy()

final_input_data["timestamp_hour"] = pd.to_datetime(final_input_data["timestamp_hour"])

final_input_data = final_input_data.sort_values(
    by=["regency", "timestamp_hour"]
).reset_index(drop=True)


# =========================
# TIME FEATURES
# =========================

final_input_data["hour"] = final_input_data["timestamp_hour"].dt.hour
final_input_data["day"] = final_input_data["timestamp_hour"].dt.day
final_input_data["month"] = final_input_data["timestamp_hour"].dt.month
final_input_data["dayofweek"] = final_input_data["timestamp_hour"].dt.dayofweek


# =========================
# SEASON FEATURE
# Bali / Indonesia:
# April - Oktober  = kemarau = 0
# November - Maret = hujan   = 1
# =========================

def get_season_binary(month):
    if month in [4, 5, 6, 7, 8, 9, 10]:
        return 0
    else:
        return 1

final_input_data["season"] = final_input_data["month"].apply(get_season_binary)


# =========================
# LAG FEATURES
# =========================

for col in pollutant_cols:
    for lag in lag_hours:
        final_input_data[f"{col}_lag_{lag}"] = (
            final_input_data
            .groupby(["regency"])[col]
            .shift(lag)
        )


# =========================
# ROLLING AVERAGE FEATURES
# shift(1) agar tidak memakai nilai jam yang sama
# =========================

for col in pollutant_cols:
    for window in rolling_windows:
        final_input_data[f"{col}_rolling_avg_{window}"] = (
            final_input_data
            .groupby(["regency"])[col]
            .transform(
                lambda x: x.shift(1).rolling(
                    window=window,
                    min_periods=1
                ).mean()
            )
        )


print("Feature engineering selesai.")
print("Total kolom:", len(final_input_data.columns))
display(final_input_data.head())

Feature engineering selesai.
Total kolom: 90


,timestamp_hour,regency,co,no,no2,o3,so2,pm2_5,pm10,nh3,...,so2_rolling_avg_24,pm2_5_rolling_avg_3,pm2_5_rolling_avg_6,pm2_5_rolling_avg_24,pm10_rolling_avg_3,pm10_rolling_avg_6,pm10_rolling_avg_24,nh3_rolling_avg_3,nh3_rolling_avg_6,nh3_rolling_avg_24
0,2026-06-22 08:00:00,Kabupaten Badung,69.545769,0.009103,0.089615,35.907564,0.155000,4.116667,12.202564,0.104359,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-22 09:00:00,Kabupaten Badung,70.414744,0.015513,0.079615,35.682436,0.153462,4.049744,11.816410,0.121410,...,0.155000,4.116667,4.116667,4.116667,12.202564,12.202564,12.202564,0.104359,0.104359,0.104359
2,2026-06-22 10:00:00,Kabupaten Badung,71.538974,0.019103,0.074744,35.731923,0.149103,4.006282,11.563333,0.157564,...,0.154231,4.083205,4.083205,4.083205,12.009487,12.009487,12.009487,0.112885,0.112885,0.112885
3,2026-06-22 11:00:00,Kabupaten Badung,72.566154,0.019103,0.069615,35.791667,0.147051,4.044103,11.876538,0.199615,...,0.152521,4.057564,4.057564,4.057564,11.860769,11.860769,11.860769,0.127778,0.127778,0.127778
4,2026-06-22 12:00:00,Kabupaten Badung,73.736282,0.019103,0.069103,35.991282,0.147692,4.087179,12.535513,0.260769,...,0.151154,4.033376,4.054199,4.054199,11.752094,11.864712,11.864712,0.159530,0.145737,0.145737


In [40]:
# =========================
# GUNAKAN SEMUA TIMESTAMP 24 JAM TERAKHIR
# SEBAGAI INPUT MODEL
# =========================

district_prediction_input = final_input_data.copy()

print("Total baris input prediksi:", len(district_prediction_input))
print("Rentang timestamp:")
print(district_prediction_input["timestamp_hour"].min())
print(district_prediction_input["timestamp_hour"].max())

display(district_prediction_input.head())

Total baris input prediksi: 192
Rentang timestamp:
2026-06-22 08:00:00
2026-06-23 07:00:00


,timestamp_hour,regency,co,no,no2,o3,so2,pm2_5,pm10,nh3,...,so2_rolling_avg_24,pm2_5_rolling_avg_3,pm2_5_rolling_avg_6,pm2_5_rolling_avg_24,pm10_rolling_avg_3,pm10_rolling_avg_6,pm10_rolling_avg_24,nh3_rolling_avg_3,nh3_rolling_avg_6,nh3_rolling_avg_24
0,2026-06-22 08:00:00,Kabupaten Badung,69.545769,0.009103,0.089615,35.907564,0.155000,4.116667,12.202564,0.104359,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-22 09:00:00,Kabupaten Badung,70.414744,0.015513,0.079615,35.682436,0.153462,4.049744,11.816410,0.121410,...,0.155000,4.116667,4.116667,4.116667,12.202564,12.202564,12.202564,0.104359,0.104359,0.104359
2,2026-06-22 10:00:00,Kabupaten Badung,71.538974,0.019103,0.074744,35.731923,0.149103,4.006282,11.563333,0.157564,...,0.154231,4.083205,4.083205,4.083205,12.009487,12.009487,12.009487,0.112885,0.112885,0.112885
3,2026-06-22 11:00:00,Kabupaten Badung,72.566154,0.019103,0.069615,35.791667,0.147051,4.044103,11.876538,0.199615,...,0.152521,4.057564,4.057564,4.057564,11.860769,11.860769,11.860769,0.127778,0.127778,0.127778
4,2026-06-22 12:00:00,Kabupaten Badung,73.736282,0.019103,0.069103,35.991282,0.147692,4.087179,12.535513,0.260769,...,0.151154,4.033376,4.054199,4.054199,11.752094,11.864712,11.864712,0.159530,0.145737,0.145737


In [41]:
# =========================
# LOAD METADATA MODEL
# =========================

metadata_path = os.path.join(MODEL_DIR, "metadata.json")

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

feature_names = metadata["feature_names"]
target_pollutants = metadata["pollutant_cols"]

print("Jumlah fitur model:", len(feature_names))
print("Target polutan:", target_pollutants)

print("\nFitur model:")
print(feature_names)

Jumlah fitur model: 89
Target polutan: ['co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']

Fitur model:
['temperature_2m', 'relative_humidity_2m', 'apparent_temperature', 'precipitation', 'rain', 'weather_code', 'cloud_cover', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'hour', 'dayofweek', 'day', 'month', 'is_weekend', 'season', 'co_lag_1', 'co_lag_3', 'co_lag_6', 'co_lag_12', 'co_lag_24', 'no_lag_1', 'no_lag_3', 'no_lag_6', 'no_lag_12', 'no_lag_24', 'no2_lag_1', 'no2_lag_3', 'no2_lag_6', 'no2_lag_12', 'no2_lag_24', 'o3_lag_1', 'o3_lag_3', 'o3_lag_6', 'o3_lag_12', 'o3_lag_24', 'so2_lag_1', 'so2_lag_3', 'so2_lag_6', 'so2_lag_12', 'so2_lag_24', 'pm2_5_lag_1', 'pm2_5_lag_3', 'pm2_5_lag_6', 'pm2_5_lag_12', 'pm2_5_lag_24', 'pm10_lag_1', 'pm10_lag_3', 'pm10_lag_6', 'pm10_lag_12', 'pm10_lag_24', 'nh3_lag_1', 'nh3_lag_3', 'nh3_lag_6', 'nh3_lag_12', 'nh3_lag_24', 'co_rolling_avg_3', 'co_rolling_avg_6', 'co_rolling_avg_24', 'no_rolling_avg_3', 'no_rolling

In [42]:
# =========================
# SIAPKAN INPUT SESUAI FITUR TRAINING
# =========================

model_input_df = district_prediction_input.copy()

missing_features = [
    col for col in feature_names
    if col not in model_input_df.columns
]

if missing_features:
    print("PERINGATAN: fitur berikut tidak tersedia di input hasil scraping:")
    print(missing_features)

    print("\nFitur tersebut akan diisi 0 sementara.")
    print("Sebaiknya pastikan fitur saat training sama dengan fitur saat prediksi.")

    for col in missing_features:
        model_input_df[col] = 0


# Isi NaN pada kolom numerik
numeric_cols = model_input_df.select_dtypes(include=[np.number]).columns

model_input_df[numeric_cols] = model_input_df[numeric_cols].fillna(
    model_input_df[numeric_cols].mean()
)

model_input_df[numeric_cols] = model_input_df[numeric_cols].fillna(0)


# Pastikan urutan kolom sama dengan training
X_new = model_input_df[feature_names]

print("Input model siap.")
print("Shape X_new:", X_new.shape)

display(X_new.head())

PERINGATAN: fitur berikut tidak tersedia di input hasil scraping:
['is_weekend', 'regency_Kabupaten Badung', 'regency_Kabupaten Bangli', 'regency_Kabupaten Buleleng', 'regency_Kabupaten Gianyar', 'regency_Kabupaten Jembrana', 'regency_Kabupaten Karangasem', 'regency_Kabupaten Klungkung', 'regency_Kota Denpasar']

Fitur tersebut akan diisi 0 sementara.
Sebaiknya pastikan fitur saat training sama dengan fitur saat prediksi.
Input model siap.
Shape X_new: (192, 89)


,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,cloud_cover,surface_pressure,wind_speed_10m,wind_direction_10m,...,nh3_rolling_avg_6,nh3_rolling_avg_24,regency_Kabupaten Badung,regency_Kabupaten Bangli,regency_Kabupaten Buleleng,regency_Kabupaten Gianyar,regency_Kabupaten Jembrana,regency_Kabupaten Karangasem,regency_Kabupaten Klungkung,regency_Kota Denpasar
0,25.138462,85.064103,29.188462,0.000000,0.000000,0.102564,10.653846,1001.288462,7.547436,68.487179,...,0.278650,0.313059,0,0,0,0,0,0,0,0
1,27.202564,77.012821,31.428205,0.000000,0.000000,0.000000,6.230769,1001.739744,7.730769,91.769231,...,0.104359,0.104359,0,0,0,0,0,0,0,0
2,28.592308,71.012821,32.767949,0.000000,0.000000,0.000000,2.858974,1001.308974,8.211538,111.076923,...,0.112885,0.112885,0,0,0,0,0,0,0,0
3,29.558974,65.717949,34.358974,0.010256,0.000000,5.294872,7.410256,1000.838462,8.307692,123.692308,...,0.127778,0.127778,0,0,0,0,0,0,0,0
4,30.026923,63.871795,34.910256,0.051282,0.001282,8.500000,9.948718,999.908974,10.439744,138.217949,...,0.145737,0.145737,0,0,0,0,0,0,0,0


In [43]:
# =========================
# LOAD MODEL TERBAIK
# PREDIKSI
# SIMPAN HASIL PREDIKSI SAJA
# =========================

prediction_result = district_prediction_input[
    ["timestamp_hour", "regency"]
].copy()

# timestamp_hour = waktu input
prediction_result = prediction_result.rename(
    columns={
        "timestamp_hour": "input_timestamp_hour"
    }
)

# Karena model memprediksi target_24h,
# maka waktu prediksinya adalah 24 jam setelah input_timestamp_hour
prediction_result["prediction_timestamp_24h"] = (
    prediction_result["input_timestamp_hour"] + pd.Timedelta(hours=24)
)

for pollutant in target_pollutants:
    model_path = os.path.join(MODEL_DIR, f"{pollutant}_best_model.joblib")

    if not os.path.exists(model_path):
        print(f"Model untuk {pollutant} tidak ditemukan: {model_path}")
        continue

    model = joblib.load(model_path)

    prediction_result[f"{pollutant}_pred_24h"] = model.predict(X_new)


# Susun urutan kolom
prediction_cols = [
    "input_timestamp_hour",
    "prediction_timestamp_24h",
    "regency",
] + [
    f"{pollutant}_pred_24h"
    for pollutant in target_pollutants
]

prediction_result = prediction_result[prediction_cols]


output_filename = "prediction_perkecamatan_24h.csv"

prediction_result.to_csv(output_filename, index=False)

print("Prediksi selesai.")
print("Hasil prediksi berhasil disimpan.")
print("Nama file:", output_filename)
print("Total baris:", len(prediction_result))

display(prediction_result)

Prediksi selesai.
Hasil prediksi berhasil disimpan.
Nama file: prediction_perkecamatan_24h.csv
Total baris: 192


,input_timestamp_hour,prediction_timestamp_24h,regency,co_pred_24h,no_pred_24h,no2_pred_24h,o3_pred_24h,so2_pred_24h,pm2_5_pred_24h,pm10_pred_24h,nh3_pred_24h
0,2026-06-22 08:00:00,2026-06-23 08:00:00,Kabupaten Badung,94.372683,0.007697,0.125395,43.818767,0.380695,9.723583,17.584574,0.207642
1,2026-06-22 09:00:00,2026-06-23 09:00:00,Kabupaten Badung,91.661118,0.011254,0.077155,43.100316,0.253071,6.531591,17.063533,0.527204
2,2026-06-22 10:00:00,2026-06-23 10:00:00,Kabupaten Badung,98.557693,0.012332,0.085124,42.518395,0.246108,6.333642,17.514024,0.568381
3,2026-06-22 11:00:00,2026-06-23 11:00:00,Kabupaten Badung,92.356295,0.016966,0.083355,42.680679,0.223459,6.568129,17.184812,0.376176
4,2026-06-22 12:00:00,2026-06-23 12:00:00,Kabupaten Badung,91.922213,0.017053,0.087167,42.752025,0.220151,6.726876,17.088470,0.336404
...,...,...,...,...,...,...,...,...,...,...,...
187,2026-06-23 03:00:00,2026-06-24 03:00:00,Kota Denpasar,176.820531,0.012373,0.420923,43.089398,0.192157,6.959008,18.761978,0.590092
188,2026-06-23 04:00:00,2026-06-24 04:00:00,Kota Denpasar,148.469980,0.012408,0.375939,43.071562,0.236752,6.795396,20.516915,0.499806
189,2026-06-23 05:00:00,2026-06-24 05:00:00,Kota Denpasar,126.000442,0.012081,0.153477,43.280182,0.181804,6.748919,20.299017,0.557010
190,2026-06-23 06:00:00,2026-06-24 06:00:00,Kota Denpasar,140.227798,0.012164,0.225239,43.046943,0.183044,6.698176,20.229391,0.557314


In [44]:
# =========================
# FUNGSI KONVERSI SATUAN DAN BREAKPOINT AQI
# =========================

MW = {
    "co": 28.01,
    "no2": 46.0055,
    "o3": 48.00,
    "so2": 64.066,
}

def ugm3_to_ppb(x, mw):
    return x * 24.45 / mw

def ugm3_to_ppm(x, mw):
    return x * 24.45 / (mw * 1000)


# =========================
# BREAKPOINT AQI 0-500
# Format: C_low, C_high, I_low, I_high
# =========================

BP_PM25 = [
    (0.0, 12.0, 0, 50),
    (12.1, 35.4, 51, 100),
    (35.5, 55.4, 101, 150),
    (55.5, 150.4, 151, 200),
    (150.5, 250.4, 201, 300),
    (250.5, 350.4, 301, 400),
    (350.5, 500.4, 401, 500),
]

BP_PM10 = [
    (0, 54, 0, 50),
    (55, 154, 51, 100),
    (155, 254, 101, 150),
    (255, 354, 151, 200),
    (355, 424, 201, 300),
    (425, 504, 301, 400),
    (505, 604, 401, 500),
]

BP_CO = [
    (0.0, 4.4, 0, 50),
    (4.5, 9.4, 51, 100),
    (9.5, 12.4, 101, 150),
    (12.5, 15.4, 151, 200),
    (15.5, 30.4, 201, 300),
    (30.5, 40.4, 301, 400),
    (40.5, 50.4, 401, 500),
]

BP_O3_8H = [
    (0.000, 0.054, 0, 50),
    (0.055, 0.070, 51, 100),
    (0.071, 0.085, 101, 150),
    (0.086, 0.105, 151, 200),
    (0.106, 0.200, 201, 300),
]

BP_O3_1H = [
    (0.125, 0.164, 101, 150),
    (0.165, 0.204, 151, 200),
    (0.205, 0.404, 201, 300),
    (0.405, 0.504, 301, 400),
    (0.505, 0.604, 401, 500),
]

BP_SO2 = [
    (0, 35, 0, 50),
    (36, 75, 51, 100),
    (76, 185, 101, 150),
    (186, 304, 151, 200),
    (305, 604, 201, 300),
    (605, 804, 301, 400),
    (805, 1004, 401, 500),
]

BP_NO2 = [
    (0, 53, 0, 50),
    (54, 100, 51, 100),
    (101, 360, 101, 150),
    (361, 649, 151, 200),
    (650, 1249, 201, 300),
    (1250, 1649, 301, 400),
    (1650, 2049, 401, 500),
]


def calc_aqi(C, breakpoints):
    if pd.isna(C):
        return np.nan

    for c_low, c_high, i_low, i_high in breakpoints:
        if c_low <= C <= c_high:
            return round(
                ((i_high - i_low) / (c_high - c_low)) * (C - c_low) + i_low
            )

    return np.nan


def aqi_category(aqi):
    if pd.isna(aqi):
        return np.nan
    elif aqi <= 50:
        return "Good"
    elif aqi <= 100:
        return "Moderate"
    elif aqi <= 150:
        return "Unhealthy for Sensitive Groups"
    elif aqi <= 200:
        return "Unhealthy"
    elif aqi <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"

print("Fungsi AQI dan breakpoint berhasil dibuat.")

Fungsi AQI dan breakpoint berhasil dibuat.


In [45]:
# =========================
# HITUNG AQI INDEX DARI HASIL PREDIKSI 24 JAM
# =========================

aqi_prediction_df = prediction_result.copy()

# Ubah kolom prediksi menjadi nama polutan asli
# supaya rumus AQI bisa memakai nama: co, no2, o3, so2, pm2_5, pm10
for pollutant in target_pollutants:
    pred_col = f"{pollutant}_pred_24h"

    if pred_col in aqi_prediction_df.columns:
        aqi_prediction_df[pollutant] = aqi_prediction_df[pred_col]


# Pastikan kolom yang dibutuhkan untuk AQI tersedia
required_aqi_pollutants = [
    "co", "no2", "o3", "so2", "pm2_5", "pm10"
]

missing_aqi_pollutants = [
    col for col in required_aqi_pollutants
    if col not in aqi_prediction_df.columns
]

if missing_aqi_pollutants:
    print("AQI tidak bisa dihitung lengkap karena kolom berikut tidak ada:")
    print(missing_aqi_pollutants)

    prediction_result["aqi_score_pred_24h"] = np.nan
    prediction_result["dominant_pollutant_pred_24h"] = np.nan
    prediction_result["aqi_index_pred_24h"] = np.nan

else:
    # =========================
    # KONVERSI SATUAN GAS
    # OpenWeather: µg/m3
    # AQI EPA gas: ppb / ppm
    # =========================

    aqi_prediction_df["co_ppm"] = ugm3_to_ppm(
        aqi_prediction_df["co"],
        MW["co"]
    )

    aqi_prediction_df["no2_ppb"] = ugm3_to_ppb(
        aqi_prediction_df["no2"],
        MW["no2"]
    )

    aqi_prediction_df["o3_ppb"] = ugm3_to_ppb(
        aqi_prediction_df["o3"],
        MW["o3"]
    )

    aqi_prediction_df["so2_ppb"] = ugm3_to_ppb(
        aqi_prediction_df["so2"],
        MW["so2"]
    )

    # O3 EPA memakai ppm
    aqi_prediction_df["o3_ppm"] = aqi_prediction_df["o3_ppb"] / 1000

    # =========================
    # AQI PER POLUTAN
    # =========================

    aqi_prediction_df["aqi_pm2_5"] = aqi_prediction_df["pm2_5"].apply(
        lambda x: calc_aqi(x, BP_PM25)
    )

    aqi_prediction_df["aqi_pm10"] = aqi_prediction_df["pm10"].apply(
        lambda x: calc_aqi(x, BP_PM10)
    )

    aqi_prediction_df["aqi_co"] = aqi_prediction_df["co_ppm"].apply(
        lambda x: calc_aqi(x, BP_CO)
    )

    aqi_prediction_df["aqi_o3_8h"] = aqi_prediction_df["o3_ppm"].apply(
        lambda x: calc_aqi(x, BP_O3_8H)
    )

    aqi_prediction_df["aqi_o3_1h"] = aqi_prediction_df["o3_ppm"].apply(
        lambda x: calc_aqi(x, BP_O3_1H)
    )

    # Untuk O3, ambil nilai lebih tinggi dari 8-hour dan 1-hour
    aqi_prediction_df["aqi_o3"] = aqi_prediction_df[
        ["aqi_o3_8h", "aqi_o3_1h"]
    ].max(axis=1)

    aqi_prediction_df["aqi_so2"] = aqi_prediction_df["so2_ppb"].apply(
        lambda x: calc_aqi(x, BP_SO2)
    )

    aqi_prediction_df["aqi_no2"] = aqi_prediction_df["no2_ppb"].apply(
        lambda x: calc_aqi(x, BP_NO2)
    )

    aqi_cols = [
        "aqi_pm2_5",
        "aqi_pm10",
        "aqi_co",
        "aqi_o3",
        "aqi_so2",
        "aqi_no2"
    ]

    # AQI final = nilai AQI tertinggi dari seluruh polutan
    aqi_prediction_df["aqi_score_pred_24h"] = (
        aqi_prediction_df[aqi_cols].max(axis=1)
    )

    aqi_prediction_df["dominant_pollutant_pred_24h"] = (
        aqi_prediction_df[aqi_cols]
        .idxmax(axis=1)
        .str.replace("aqi_", "", regex=False)
    )

    aqi_prediction_df["aqi_index_pred_24h"] = (
        aqi_prediction_df["aqi_score_pred_24h"].apply(aqi_category)
    )

    # Tambahkan hasil AQI ke prediction_result
    prediction_result["aqi_score_pred_24h"] = (
        aqi_prediction_df["aqi_score_pred_24h"]
    )

    prediction_result["dominant_pollutant_pred_24h"] = (
        aqi_prediction_df["dominant_pollutant_pred_24h"]
    )

    prediction_result["aqi_index_pred_24h"] = (
        aqi_prediction_df["aqi_index_pred_24h"]
    )


# Susun ulang kolom akhir
prediction_cols_with_aqi = [
    "input_timestamp_hour",
    "prediction_timestamp_24h",
    "regency",
] + [
    f"{pollutant}_pred_24h"
    for pollutant in target_pollutants
] + [
    "aqi_score_pred_24h",
    "dominant_pollutant_pred_24h",
    "aqi_index_pred_24h"
]

prediction_cols_with_aqi = [
    col for col in prediction_cols_with_aqi
    if col in prediction_result.columns
]

prediction_result_with_aqi = prediction_result[prediction_cols_with_aqi].copy()

output_filename_aqi = "prediction_perkabupaten_24h_with_aqi.csv"

prediction_result_with_aqi.to_csv(output_filename_aqi, index=False)

print("Perhitungan AQI selesai.")
print("Hasil prediksi dengan AQI berhasil disimpan.")
print("Nama file:", output_filename_aqi)
print("Total baris:", len(prediction_result_with_aqi))

display(prediction_result_with_aqi)

Perhitungan AQI selesai.
Hasil prediksi dengan AQI berhasil disimpan.
Nama file: prediction_perkabupaten_24h_with_aqi.csv
Total baris: 192


,input_timestamp_hour,prediction_timestamp_24h,regency,co_pred_24h,no_pred_24h,no2_pred_24h,o3_pred_24h,so2_pred_24h,pm2_5_pred_24h,pm10_pred_24h,nh3_pred_24h,aqi_score_pred_24h,dominant_pollutant_pred_24h,aqi_index_pred_24h
0,2026-06-22 08:00:00,2026-06-23 08:00:00,Kabupaten Badung,94.372683,0.007697,0.125395,43.818767,0.380695,9.723583,17.584574,0.207642,41.0,pm2_5,Good
1,2026-06-22 09:00:00,2026-06-23 09:00:00,Kabupaten Badung,91.661118,0.011254,0.077155,43.100316,0.253071,6.531591,17.063533,0.527204,27.0,pm2_5,Good
2,2026-06-22 10:00:00,2026-06-23 10:00:00,Kabupaten Badung,98.557693,0.012332,0.085124,42.518395,0.246108,6.333642,17.514024,0.568381,26.0,pm2_5,Good
3,2026-06-22 11:00:00,2026-06-23 11:00:00,Kabupaten Badung,92.356295,0.016966,0.083355,42.680679,0.223459,6.568129,17.184812,0.376176,27.0,pm2_5,Good
4,2026-06-22 12:00:00,2026-06-23 12:00:00,Kabupaten Badung,91.922213,0.017053,0.087167,42.752025,0.220151,6.726876,17.088470,0.336404,28.0,pm2_5,Good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,2026-06-23 03:00:00,2026-06-24 03:00:00,Kota Denpasar,176.820531,0.012373,0.420923,43.089398,0.192157,6.959008,18.761978,0.590092,29.0,pm2_5,Good
188,2026-06-23 04:00:00,2026-06-24 04:00:00,Kota Denpasar,148.469980,0.012408,0.375939,43.071562,0.236752,6.795396,20.516915,0.499806,28.0,pm2_5,Good
189,2026-06-23 05:00:00,2026-06-24 05:00:00,Kota Denpasar,126.000442,0.012081,0.153477,43.280182,0.181804,6.748919,20.299017,0.557010,28.0,pm2_5,Good
190,2026-06-23 06:00:00,2026-06-24 06:00:00,Kota Denpasar,140.227798,0.012164,0.225239,43.046943,0.183044,6.698176,20.229391,0.557314,28.0,pm2_5,Good
